
# Phase 2 — Data Cleaning and Quality Audit

## Mission 01: Data Quality Audit

**Dataset:** October 2019 eCommerce Customer Sample

**Expected rows:** 849,220

**Objective:** Identify and document data quality issues before applying cleaning transformations.

**Important:** Preserve the original analytical sample. All cleaning decisions must be justified and reproducible.


In [1]:

import pandas as pd
from pathlib import Path

# Locate the project root from the notebook
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Path to our existing analytical sample
sample_path = (
    project_root
    / "data"
    / "raw"
    / "october_customer_sample.csv"
)

# Load the sample without modifying the source CSV
df = pd.read_csv(sample_path)

# Basic verification
print("========== PHASE 2: DATASET LOAD ==========")
print(f"Dataset path: {sample_path}")
print(f"Total rows: {len(df):,}")
print(f"Total columns: {df.shape[1]}")
print(f"Expected row count matched: {len(df) == 849220}")

print("\n========== COLUMN NAMES ==========")
print(df.columns.tolist())

print("\n========== FIRST 5 ROWS ==========")
display(df.head())


========== PHASE 2: DATASET LOAD ==========
Dataset path: c:\Users\kunal\OneDrive\Desktop\product-funnel-analysis\data\raw\october_customer_sample.csv
Total rows: 849,220
Total columns: 9
Expected row count matched: True

========== COLUMN NAMES ==========
['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

========== FIRST 5 ROWS ==========


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:56 UTC,view,4804295,2053013554658804075,electronics.audio.headphone,xiaomi,23.13,541366014,6d8ce6fb-2953-4e95-96fe-b40f1ec50ba8
1,2019-10-01 00:03:35 UTC,view,1005115,2053013555631882655,electronics.smartphone,apple,975.57,519829701,c3012f56-70f3-4198-960d-5f14467c0a53
2,2019-10-01 00:07:43 UTC,view,15100370,2053013557024391671,NaN,NaN,257.15,527776076,b1185e33-79ce-4fcb-b85a-d06aeb87beee
3,2019-10-01 00:08:33 UTC,view,15100350,2053013557024391671,NaN,NaN,321.73,527776076,b1185e33-79ce-4fcb-b85a-d06aeb87beee
4,2019-10-01 00:09:19 UTC,view,15100100,2053013557024391671,NaN,NaN,437.33,527776076,b1185e33-79ce-4fcb-b85a-d06aeb87beee



## Step 2 — Missing Value Audit

Objective: Identify missing values in each column and measure their percentage of the analytical sample.

No cleaning transformations will be applied at this stage.


In [2]:

# Calculate missing values for each column
missing_count = df.isna().sum()

# Calculate missing percentages
missing_percentage = (missing_count / len(df)) * 100

# Create a data quality summary
missing_audit = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_percentage.round(2)
})

# Sort columns by missing count
missing_audit = missing_audit.sort_values(
    by="missing_count",
    ascending=False
)

print("========== MISSING VALUE AUDIT ==========")
display(missing_audit)

print("\n========== ROW-LEVEL MISSINGNESS ==========")
print(f"Rows with at least one missing value: {df.isna().any(axis=1).sum():,}")
print(f"Rows with no missing values: {df.notna().all(axis=1).sum():,}")


========== MISSING VALUE AUDIT ==========


,missing_count,missing_percentage
category_code,268555,31.62
brand,121933,14.36
event_time,0,0.00
product_id,0,0.00
event_type,0,0.00
category_id,0,0.00
price,0,0.00
user_id,0,0.00
user_session,0,0.00



========== ROW-LEVEL MISSINGNESS ==========
Rows with at least one missing value: 315,791
Rows with no missing values: 533,429



## Step 3 — Duplicate Detection

Objective: Identify exact duplicate records across all nine original dataset columns.

Important: Repeated customer or product activity does not automatically indicate invalid duplicate events.

No records will be removed during this audit.


In [3]:

# Identify exact duplicate rows
duplicate_mask = df.duplicated(keep="first")

# Calculate duplicate statistics
duplicate_count = duplicate_mask.sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print("========== EXACT DUPLICATE AUDIT ==========")
print(f"Total rows: {len(df):,}")
print(f"Exact duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")
print(f"Rows after hypothetical deduplication: {len(df) - duplicate_count:,}")

print("\n========== DUPLICATE EXAMPLES ==========")

# Preview duplicate rows without deleting anything
duplicate_examples = df[df.duplicated(keep=False)]

if duplicate_examples.empty:
    print("No exact duplicate records found.")
else:
    display(duplicate_examples.head(10))


========== EXACT DUPLICATE AUDIT ==========
Total rows: 849,220
Exact duplicate rows: 600
Duplicate percentage: 0.07%
Rows after hypothetical deduplication: 848,620

========== DUPLICATE EXAMPLES ==========


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
31348,2019-10-02 07:28:47 UTC,cart,1004874,2053013555631882655,electronics.smartphone,samsung,383.51,542837325,e864be69-e104-4b74-93d1-1655795244ff
31349,2019-10-02 07:28:47 UTC,cart,1004874,2053013555631882655,electronics.smartphone,samsung,383.51,542837325,e864be69-e104-4b74-93d1-1655795244ff
34900,2019-10-02 10:09:03 UTC,cart,1004229,2053013555631882655,electronics.smartphone,apple,1143.09,513643225,6185b9d1-d9ca-423d-a014-365f4fc9be97
34901,2019-10-02 10:09:03 UTC,cart,1004229,2053013555631882655,electronics.smartphone,apple,1143.09,513643225,6185b9d1-d9ca-423d-a014-365f4fc9be97
41403,2019-10-02 15:10:41 UTC,cart,1200947,2172371436436455782,electronics.tablet,samsung,115.32,554250511,6a4e7c72-5b02-46f0-9c44-b960b27ffad0
41404,2019-10-02 15:10:41 UTC,cart,1200947,2172371436436455782,electronics.tablet,samsung,115.32,554250511,6a4e7c72-5b02-46f0-9c44-b960b27ffad0
41405,2019-10-02 15:10:41 UTC,cart,1200947,2172371436436455782,electronics.tablet,samsung,115.32,554250511,6a4e7c72-5b02-46f0-9c44-b960b27ffad0
41406,2019-10-02 15:10:42 UTC,cart,1200947,2172371436436455782,electronics.tablet,samsung,115.32,554250511,6a4e7c72-5b02-46f0-9c44-b960b27ffad0
41407,2019-10-02 15:10:42 UTC,cart,1200947,2172371436436455782,electronics.tablet,samsung,115.32,554250511,6a4e7c72-5b02-46f0-9c44-b960b27ffad0
46456,2019-10-02 18:23:18 UTC,cart,1004809,2053013555631882655,electronics.smartphone,xiaomi,125.55,546575246,957360f9-f849-418e-94f3-6757351907f5



## Step 4 — Data Type Validation

Objective: Inspect the current pandas data types and compare them with the expected analytical data types.

No type conversions will be performed during this audit.


In [4]:

print("========== DATA TYPE AUDIT ==========")

# Display pandas data types
print(df.dtypes)

print("\n========== DATAFRAME INFORMATION ==========")

# Display column types and non-null counts
df.info()

print("\n========== EXPECTED VS ACTUAL TYPES ==========")

expected_types = {
    "event_time": "datetime",
    "event_type": "object",
    "product_id": "integer",
    "category_id": "integer",
    "category_code": "object",
    "brand": "object",
    "price": "float",
    "user_id": "integer",
    "user_session": "object"
}

for column, expected in expected_types.items():
    actual = str(df[column].dtype)
    print(f"{column}: Actual = {actual} | Expected = {expected}")


========== DATA TYPE AUDIT ==========
event_time           str
event_type           str
product_id         int64
category_id        int64
category_code        str
brand                str
price            float64
user_id            int64
user_session         str
dtype: object

========== DATAFRAME INFORMATION ==========
<class 'pandas.DataFrame'>
RangeIndex: 849220 entries, 0 to 849219
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     849220 non-null  str    
 1   event_type     849220 non-null  str    
 2   product_id     849220 non-null  int64  
 3   category_id    849220 non-null  int64  
 4   category_code  580665 non-null  str    
 5   brand          727287 non-null  str    
 6   price          849220 non-null  float64
 7   user_id        849220 non-null  int64  
 8   user_session   849220 non-null  str    
dtypes: float64(1), int64(3), str(5)
memory usage: 58.3 MB

========== EXPECTED VS A


## Step 5 — Invalid Value Detection

Objective: Identify invalid prices, identifiers, event types, and session values.

All checks are read-only. No records will be modified or removed.


In [5]:

print("========== INVALID VALUE AUDIT ==========")

# 1. Price validation
negative_prices = (df["price"] < 0).sum()
zero_prices = (df["price"] == 0).sum()

print("\nPRICE VALIDATION")
print(f"Negative prices: {negative_prices:,}")
print(f"Zero prices: {zero_prices:,}")

# 2. Identifier validation
print("\nIDENTIFIER VALIDATION")

for column in ["product_id", "category_id", "user_id"]:
    invalid_count = (df[column] <= 0).sum()
    print(f"{column} <= 0: {invalid_count:,}")

# 3. Event type validation
print("\nEVENT TYPE VALIDATION")

valid_event_types = {"view", "cart", "purchase"}

invalid_event_types = (
    ~df["event_type"].isin(valid_event_types)
).sum()

print(f"Invalid event types: {invalid_event_types:,}")

print("\nObserved event types:")
print(df["event_type"].value_counts(dropna=False))

# 4. Session validation
print("\nSESSION VALIDATION")

blank_sessions = (
    df["user_session"].fillna("").str.strip() == ""
).sum()

print(f"Blank session IDs: {blank_sessions:,}")


========== INVALID VALUE AUDIT ==========

PRICE VALIDATION
Negative prices: 0
Zero prices: 1,469

IDENTIFIER VALIDATION
product_id <= 0: 0
category_id <= 0: 0
user_id <= 0: 0

EVENT TYPE VALIDATION
Invalid event types: 0

Observed event types:
event_type
view        814847
cart         18938
purchase     15435
Name: count, dtype: int64

SESSION VALIDATION
Blank session IDs: 0



## Step 6 — Timestamp Validation

Objective: Validate timestamp parsing, chronological boundaries, and UTC timezone consistency.

Expected period: October 1–31, 2019 (UTC).

No changes will be made to the original dataset.


In [6]:

print("========== TIMESTAMP VALIDATION ==========")

# Parse timestamps into a separate temporary Series
parsed_timestamps = pd.to_datetime(
    df["event_time"],
    errors="coerce",
    utc=True
)

# 1. Check invalid timestamps
invalid_timestamps = parsed_timestamps.isna().sum()

print(f"Invalid timestamps: {invalid_timestamps:,}")

# 2. Check date boundaries
start_date = pd.Timestamp("2019-10-01", tz="UTC")
end_date = pd.Timestamp("2019-11-01", tz="UTC")

outside_october = (
    (parsed_timestamps < start_date)
    | (parsed_timestamps >= end_date)
).sum()

print(f"Events outside October 2019: {outside_october:,}")

# 3. Minimum and maximum timestamps
print("\n========== DATE RANGE ==========")
print(f"Earliest event: {parsed_timestamps.min()}")
print(f"Latest event: {parsed_timestamps.max()}")

# 4. Check timezone
print("\n========== TIMEZONE ==========")
print(f"Parsed timezone: {parsed_timestamps.dt.tz}")

# 5. Verify number of unique dates
print("\n========== DATE COVERAGE ==========")
print(
    f"Unique event dates: "
    f"{parsed_timestamps.dt.date.nunique()}"
)

# 6. Verify original column remains unchanged
print("\n========== ORIGINAL DATA TYPE ==========")
print(f"event_time dtype: {df['event_time'].dtype}")


========== TIMESTAMP VALIDATION ==========
Invalid timestamps: 0
Events outside October 2019: 0

========== DATE RANGE ==========
Earliest event: 2019-10-01 00:00:56+00:00
Latest event: 2019-10-31 23:57:44+00:00

========== TIMEZONE ==========
Parsed timezone: UTC

========== DATE COVERAGE ==========
Unique event dates: 31

========== ORIGINAL DATA TYPE ==========
event_time dtype: str



# Mission 02 — Cleaning Rules and Decisions

## Step 1 — Missing Value Impact Investigation

Objective: Determine how missing category_code and brand values are distributed across event types.

No records will be modified or deleted.


In [7]:

print("========== MISSING VALUES BY EVENT TYPE ==========")

# Count missing category_code values by event type
missing_category = (
    df.groupby("event_type")["category_code"]
    .apply(lambda x: x.isna().sum())
)

# Count missing brand values by event type
missing_brand = (
    df.groupby("event_type")["brand"]
    .apply(lambda x: x.isna().sum())
)

# Total events by type
event_totals = df["event_type"].value_counts()

# Create summary
missing_impact = pd.DataFrame({
    "total_events": event_totals,
    "missing_category_code": missing_category,
    "missing_brand": missing_brand
})

# Calculate percentages within each event type
missing_impact["category_missing_pct"] = (
    missing_impact["missing_category_code"]
    / missing_impact["total_events"] * 100
).round(2)

missing_impact["brand_missing_pct"] = (
    missing_impact["missing_brand"]
    / missing_impact["total_events"] * 100
).round(2)

display(missing_impact)

print("\n========== PURCHASE IMPACT ==========")

purchases = df[df["event_type"] == "purchase"]

print(f"Total purchase events: {len(purchases):,}")
print(
    f"Purchases missing category_code: "
    f"{purchases['category_code'].isna().sum():,}"
)
print(
    f"Purchases missing brand: "
    f"{purchases['brand'].isna().sum():,}"
)


========== MISSING VALUES BY EVENT TYPE ==========


,total_events,missing_category_code,missing_brand,category_missing_pct,brand_missing_pct
event_type,,,,,
cart,18938,1931,409,10.20,2.16
purchase,15435,3527,1318,22.85,8.54
view,814847,263097,120206,32.29,14.75



========== PURCHASE IMPACT ==========
Total purchase events: 15,435
Purchases missing category_code: 3,527
Purchases missing brand: 1,318



## Mission 02 — Step 3: Duplicate Impact Investigation

Objective: Investigate the 600 exact duplicate candidates
by event type and assess their potential impact.

Exact duplicate records are not automatically invalid events.

No records will be removed during this investigation.


In [8]:

print("========== DUPLICATE IMPACT INVESTIGATION ==========")

# Identify duplicate records beyond their first occurrence
duplicate_mask = df.duplicated(keep="first")

# Extract duplicate candidates
duplicate_records = df.loc[duplicate_mask]

# Count duplicates by event type
duplicate_by_event = (
    duplicate_records["event_type"]
    .value_counts()
    .reindex(["view", "cart", "purchase"], fill_value=0)
)

print("\nDUPLICATES BY EVENT TYPE")
display(duplicate_by_event.to_frame("duplicate_count"))

# Verify total duplicate count
print("\nDUPLICATE RECONCILIATION")
print(f"Total duplicate candidates: {duplicate_mask.sum():,}")
print(f"Sum by event type: {duplicate_by_event.sum():,}")
print(f"Counts match: {duplicate_mask.sum() == duplicate_by_event.sum()}")

# Examine duplicate purchase records
duplicate_purchases = duplicate_records[
    duplicate_records["event_type"] == "purchase"
]

print("\nPURCHASE DUPLICATE INVESTIGATION")
print(f"Duplicate purchase candidates: {len(duplicate_purchases):,}")

if duplicate_purchases.empty:
    print("No exact duplicate purchase candidates found.")
else:
    display(duplicate_purchases.head(10))


========== DUPLICATE IMPACT INVESTIGATION ==========

DUPLICATES BY EVENT TYPE


,duplicate_count
event_type,
view,35
cart,564
purchase,1



DUPLICATE RECONCILIATION
Total duplicate candidates: 600
Sum by event type: 600
Counts match: True

PURCHASE DUPLICATE INVESTIGATION
Duplicate purchase candidates: 1


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
213390,2019-10-09 08:41:38 UTC,purchase,2601779,2053013563970159485,NaN,artel,185.08,539261575,64a012c7-0a10-460b-8f95-aa011fe8b825



## Mission 02 — Step 4: Zero-Price Investigation

Objective: Investigate zero-price events by event type,
product, and category.

Zero-price records are not automatically invalid.

No cleaning transformations will be applied.


In [9]:

print("========== ZERO-PRICE INVESTIGATION ==========")

# Identify zero-price records
zero_price_df = df[df["price"] == 0]

print(f"Total zero-price events: {len(zero_price_df):,}")

# 1. Distribution by event type
print("\nZERO-PRICE EVENTS BY EVENT TYPE")

zero_price_by_event = (
    zero_price_df["event_type"]
    .value_counts()
    .reindex(["view", "cart", "purchase"], fill_value=0)
)

display(zero_price_by_event.to_frame("zero_price_count"))

# 2. Number of affected products
print("\nAFFECTED PRODUCTS")

print(
    f"Unique products with zero-price events: "
    f"{zero_price_df['product_id'].nunique():,}"
)

# 3. Most common categories
print("\nTOP 10 CATEGORIES")

display(
    zero_price_df["category_code"]
    .fillna("Unknown")
    .value_counts()
    .head(10)
    .to_frame("zero_price_count")
)

# 4. Preview zero-price purchase events
print("\nZERO-PRICE PURCHASE INVESTIGATION")

zero_price_purchases = zero_price_df[
    zero_price_df["event_type"] == "purchase"
]

print(
    f"Zero-price purchase events: "
    f"{len(zero_price_purchases):,}"
)

if zero_price_purchases.empty:
    print("No zero-price purchases found.")
else:
    display(zero_price_purchases.head(10))


========== ZERO-PRICE INVESTIGATION ==========
Total zero-price events: 1,469

ZERO-PRICE EVENTS BY EVENT TYPE


,zero_price_count
event_type,
view,1468
cart,1
purchase,0



AFFECTED PRODUCTS
Unique products with zero-price events: 852

TOP 10 CATEGORIES


,zero_price_count
category_code,
Unknown,699
electronics.smartphone,91
apparel.shoes,91
construction.tools.welding,52
electronics.clocks,49
furniture.bedroom.bed,36
appliances.environment.vacuum,34
furniture.living_room.cabinet,32
computers.notebook,32



ZERO-PRICE PURCHASE INVESTIGATION
Zero-price purchase events: 0
No zero-price purchases found.



# Mission 03 — Cleaned Dataset Creation

## Step 1 — Create a Safe Working Copy

Objective: Create an independent working DataFrame
for applying the documented cleaning rules.

The original DataFrame and raw CSV must remain unchanged.


In [10]:

print("========== CREATE CLEANING WORKING COPY ==========")

# Create an independent copy of the original DataFrame
df_clean = df.copy(deep=True)

# Verify original and working DataFrame dimensions
print(f"Original rows: {len(df):,}")
print(f"Working copy rows: {len(df_clean):,}")

print(f"Original columns: {df.shape[1]}")
print(f"Working copy columns: {df_clean.shape[1]}")

# Verify the initial content is identical
print(f"Data identical: {df.equals(df_clean)}")

# Verify they are separate DataFrame objects
print(f"Separate objects: {df is not df_clean}")

print("\n========== WORKING COPY CREATED ==========")


========== CREATE CLEANING WORKING COPY ==========
Original rows: 849,220
Working copy rows: 849,220
Original columns: 9
Working copy columns: 9
Data identical: True
Separate objects: True

========== WORKING COPY CREATED ==========



## Step 2 — Missing Value Treatment

Objective: Replace missing category_code and brand
values with "Unknown" in the cleaned working DataFrame.

Preserve all original records and keep the raw
DataFrame unchanged.


In [11]:

print("========== MISSING VALUE TREATMENT ==========")

# Record missing values before cleaning
missing_before = df_clean[
    ["category_code", "brand"]
].isna().sum()

print("\nMISSING VALUES BEFORE CLEANING")
print(missing_before)

# Apply documented cleaning rules
df_clean["category_code"] = (
    df_clean["category_code"].fillna("Unknown")
)

df_clean["brand"] = (
    df_clean["brand"].fillna("Unknown")
)

# Verify missing values after cleaning
missing_after = df_clean[
    ["category_code", "brand"]
].isna().sum()

print("\nMISSING VALUES AFTER CLEANING")
print(missing_after)

# Confirm original DataFrame remains unchanged
print("\nORIGINAL DATAFRAME VERIFICATION")
print("Original category_code missing:", df["category_code"].isna().sum())
print("Original brand missing:", df["brand"].isna().sum())

# Confirm no rows were removed
print("\nROW COUNT VERIFICATION")
print(f"Original rows: {len(df):,}")
print(f"Cleaned rows: {len(df_clean):,}")
print(f"Row count preserved: {len(df) == len(df_clean)}")

# Confirm expected replacements
print("\nUNKNOWN VALUE COUNTS")
print("category_code:", (df_clean["category_code"] == "Unknown").sum())
print("brand:", (df_clean["brand"] == "Unknown").sum())


========== MISSING VALUE TREATMENT ==========

MISSING VALUES BEFORE CLEANING
category_code    268555
brand            121933
dtype: int64

MISSING VALUES AFTER CLEANING
category_code    0
brand            0
dtype: int64

ORIGINAL DATAFRAME VERIFICATION
Original category_code missing: 268555
Original brand missing: 121933

ROW COUNT VERIFICATION
Original rows: 849,220
Cleaned rows: 849,220
Row count preserved: True

UNKNOWN VALUE COUNTS
category_code: 268555
brand: 121933



## Step 3 — UTC Timestamp Conversion

Objective: Convert event_time from string to a
timezone-aware UTC datetime.

Preserve all records, original timestamp meanings,
and the raw DataFrame.


In [12]:

import pandas as pd

print("========== UTC TIMESTAMP CONVERSION ==========")

# Record original timestamp values
original_timestamps = pd.to_datetime(
    df["event_time"],
    utc=True,
    errors="raise"
)

# Convert timestamps in the cleaned working copy
df_clean["event_time"] = pd.to_datetime(
    df_clean["event_time"],
    utc=True,
    errors="raise"
)

print("\nDATA TYPE VERIFICATION")
print("Original dtype:", df["event_time"].dtype)
print("Cleaned dtype:", df_clean["event_time"].dtype)

print("\nTIMEZONE VERIFICATION")
print("Cleaned timezone:", df_clean["event_time"].dt.tz)

print("\nTIMESTAMP INTEGRITY")
print(
    "All timestamps preserved:",
    original_timestamps.equals(df_clean["event_time"])
)

print(
    "Invalid timestamps:",
    df_clean["event_time"].isna().sum()
)

print("\nDATE RANGE")
print("Earliest event:", df_clean["event_time"].min())
print("Latest event:", df_clean["event_time"].max())

print("\nROW COUNT VERIFICATION")
print(f"Original rows: {len(df):,}")
print(f"Cleaned rows: {len(df_clean):,}")
print("Rows preserved:", len(df) == len(df_clean))

print("\nFIRST 5 CONVERTED TIMESTAMPS")
display(df_clean[["event_time"]].head())


========== UTC TIMESTAMP CONVERSION ==========

DATA TYPE VERIFICATION
Original dtype: str
Cleaned dtype: datetime64[us, UTC]

TIMEZONE VERIFICATION
Cleaned timezone: UTC

TIMESTAMP INTEGRITY
All timestamps preserved: True
Invalid timestamps: 0

DATE RANGE
Earliest event: 2019-10-01 00:00:56+00:00
Latest event: 2019-10-31 23:57:44+00:00

ROW COUNT VERIFICATION
Original rows: 849,220
Cleaned rows: 849,220
Rows preserved: True

FIRST 5 CONVERTED TIMESTAMPS


,event_time
0,2019-10-01 00:00:56+00:00
1,2019-10-01 00:03:35+00:00
2,2019-10-01 00:07:43+00:00
3,2019-10-01 00:08:33+00:00
4,2019-10-01 00:09:19+00:00



## Step 4 — Cleaning Validation

Objective: Verify that all cleaning transformations
follow the documented rules.

Validate record counts, missing values, duplicate
retention, zero-price retention, identifiers,
and timestamp integrity.

Primary funnel KPI reconciliation will be
performed in Mission 04.


In [13]:

print("========== CLEANING VALIDATION ==========")

# 1. Row and column preservation
print("\nSTRUCTURE VALIDATION")

print("Rows preserved:", len(df) == len(df_clean))
print("Columns preserved:", df.columns.equals(df_clean.columns))

# 2. Missing value validation
print("\nMISSING VALUE VALIDATION")

print("Missing category_code:", df_clean["category_code"].isna().sum())
print("Missing brand:", df_clean["brand"].isna().sum())

# 3. Duplicate retention
print("\nDUPLICATE VALIDATION")

print("Original duplicates:", df.duplicated().sum())
print("Cleaned duplicates:", df_clean.duplicated().sum())

# 4. Zero-price retention
print("\nZERO-PRICE VALIDATION")

print("Original zero prices:", (df["price"] == 0).sum())
print("Cleaned zero prices:", (df_clean["price"] == 0).sum())

# 5. Critical column integrity
print("\nCRITICAL COLUMN VALIDATION")

critical_columns = [
    "user_id",
    "user_session",
    "product_id",
    "event_type",
    "price"
]

for column in critical_columns:
    print(
        f"{column} preserved:",
        df[column].equals(df_clean[column])
    )

# 6. Timestamp integrity
print("\nTIMESTAMP VALIDATION")

print(
    "Timestamps preserved:",
    pd.to_datetime(
        df["event_time"],
        utc=True,
        errors="raise"
    ).equals(df_clean["event_time"])
)

# 7. Final dataset shape
print("\nFINAL DATASET SHAPE")

print("Rows:", f"{df_clean.shape[0]:,}")
print("Columns:", df_clean.shape[1])

print("\n========== VALIDATION FINISHED ==========")


========== CLEANING VALIDATION ==========

STRUCTURE VALIDATION
Rows preserved: True
Columns preserved: True

MISSING VALUE VALIDATION
Missing category_code: 0
Missing brand: 0

DUPLICATE VALIDATION
Original duplicates: 600
Cleaned duplicates: 600

ZERO-PRICE VALIDATION
Original zero prices: 1469
Cleaned zero prices: 1469

CRITICAL COLUMN VALIDATION
user_id preserved: True
user_session preserved: True
product_id preserved: True
event_type preserved: True
price preserved: True

TIMESTAMP VALIDATION
Timestamps preserved: True

FINAL DATASET SHAPE
Rows: 849,220
Columns: 9

========== VALIDATION FINISHED ==========


In [14]:

print("========== FINAL TIMESTAMP CHECK ==========")

timestamp_match = pd.to_datetime(
    df["event_time"],
    utc=True,
    errors="raise"
).equals(df_clean["event_time"])

print("Timestamps preserved:", timestamp_match)
print("Timezone:", df_clean["event_time"].dt.tz)
print("Total rows:", f"{len(df_clean):,}")

assert timestamp_match, "Timestamp integrity check failed"
assert len(df_clean) == 849220, "Unexpected row count"

print("Final timestamp validation: PASSED")


========== FINAL TIMESTAMP CHECK ==========
Timestamps preserved: True
Timezone: UTC
Total rows: 849,220
Final timestamp validation: PASSED



## Step 5 — Export Cleaned Dataset

Objective: Export the validated cleaned analytical
dataset to the data/clean directory.

Preserve the original raw CSV.

The exported file will be used for PostgreSQL
loading and downstream analysis.


In [15]:

from pathlib import Path

print("========== EXPORT CLEANED DATASET ==========")

# Locate the project root
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Define output directory and file
clean_dir = project_root / "data" / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)

clean_file = clean_dir / "october_customer_sample_clean.csv"

# Export the cleaned dataset
df_clean.to_csv(
    clean_file,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S%z"
)

print("\nEXPORT DETAILS")
print("Output file:", clean_file)
print("File exists:", clean_file.exists())
print("Exported rows:", f"{len(df_clean):,}")
print("Exported columns:", df_clean.shape[1])

print("\nFILE SIZE")
print(
    "Size (MB):",
    round(clean_file.stat().st_size / (1024 ** 2), 2)
)

print("\n========== EXPORT COMPLETE ==========")


========== EXPORT CLEANED DATASET ==========

EXPORT DETAILS
Output file: c:\Users\kunal\OneDrive\Desktop\product-funnel-analysis\data\clean\october_customer_sample_clean.csv
File exists: True
Exported rows: 849,220
Exported columns: 9

FILE SIZE
Size (MB): 112.37

========== EXPORT COMPLETE ==========



## Step 6 — CSV Read-Back Verification

Objective: Reload the exported cleaned CSV and verify
its structure, data integrity, and UTC timestamps.

Confirm that the exported file is suitable for
downstream PostgreSQL preparation.


In [16]:

import pandas as pd

print("========== CSV READ-BACK VERIFICATION ==========")

# Reload the exported CSV with explicit data types
df_reloaded = pd.read_csv(
    clean_file,
    dtype={
        "product_id": "int64",
        "category_id": "int64",
        "user_id": "int64",
        "event_type": "str",
        "category_code": "str",
        "brand": "str",
        "user_session": "str"
    },
    parse_dates=["event_time"]
)

print("\nSTRUCTURE VALIDATION")
print("Reloaded rows:", f"{len(df_reloaded):,}")
print("Reloaded columns:", df_reloaded.shape[1])
print("Rows match:", len(df_reloaded) == len(df_clean))
print("Columns match:", df_reloaded.columns.equals(df_clean.columns))

print("\nMISSING VALUE VALIDATION")
print("Missing category_code:", df_reloaded["category_code"].isna().sum())
print("Missing brand:", df_reloaded["brand"].isna().sum())

print("\nTIMESTAMP VALIDATION")
print("Reloaded dtype:", df_reloaded["event_time"].dtype)
print("Reloaded timezone:", df_reloaded["event_time"].dt.tz)

print(
    "Timestamps match:",
    df_reloaded["event_time"].equals(
        df_clean["event_time"]
    )
)

print("\nBUSINESS RULE VALIDATION")
print("Exact duplicates:", df_reloaded.duplicated().sum())
print("Zero-price events:", (df_reloaded["price"] == 0).sum())

print("\nDATA INTEGRITY")
print(
    "All values match:",
    df_reloaded.equals(df_clean)
)

print("\n========== READ-BACK VERIFICATION FINISHED ==========")


========== CSV READ-BACK VERIFICATION ==========

STRUCTURE VALIDATION
Reloaded rows: 849,220
Reloaded columns: 9
Rows match: True
Columns match: True

MISSING VALUE VALIDATION
Missing category_code: 0
Missing brand: 0

TIMESTAMP VALIDATION
Reloaded dtype: datetime64[us, UTC]
Reloaded timezone: UTC
Timestamps match: True

BUSINESS RULE VALIDATION
Exact duplicates: 600
Zero-price events: 1469

DATA INTEGRITY
All values match: True

========== READ-BACK VERIFICATION FINISHED ==========



---

# Mission 04 — Funnel Reconciliation and Final Validation

## Step 1 — Establish Phase 1 Funnel Baseline

Compare the cleaned dataset against the original Phase 1
journey-level funnel metrics.

Primary journey key:
user_id + user_session + product_id

Required chronological sequence:
view → cart → purchase


In [1]:

# Phase 1 verified journey-level funnel baseline

phase1_baseline = {
    "view_journeys": 552900,
    "view_to_cart_journeys": 12725,
    "completed_journeys": 6300
}

print("========== PHASE 1 FUNNEL BASELINE ==========")

for metric, value in phase1_baseline.items():
    print(f"{metric}: {value:,}")

print("\nBaseline established successfully.")


========== PHASE 1 FUNNEL BASELINE ==========
view_journeys: 552,900
view_to_cart_journeys: 12,725
completed_journeys: 6,300

Baseline established successfully.


In [3]:

import pandas as pd
from pathlib import Path

# Locate the project directory
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Locate the cleaned dataset
clean_file = (
    project_root
    / "data"
    / "clean"
    / "october_customer_sample_clean.csv"
)

# Load cleaned dataset with UTC timestamps
df_clean = pd.read_csv(
    clean_file,
    parse_dates=["event_time"]
)

# Verify loading
print("========== CLEANED DATASET RELOADED ==========")
print(f"Rows: {len(df_clean):,}")
print(f"Columns: {len(df_clean.columns)}")
print(f"Timestamp dtype: {df_clean['event_time'].dtype}")
print(f"File exists: {clean_file.exists()}")


========== CLEANED DATASET RELOADED ==========
Rows: 849,220
Columns: 9
Timestamp dtype: datetime64[us, UTC]
File exists: True


In [4]:

print("========== FUNNEL INPUT VALIDATION ==========")

required_columns = [
    "user_id",
    "user_session",
    "product_id",
    "event_type",
    "event_time"
]

print("\nREQUIRED COLUMNS")
for col in required_columns:
    print(f"{col}: {col in df_clean.columns}")

print("\nDATASET STRUCTURE")
print(f"Rows: {len(df_clean):,}")
print(f"Columns: {len(df_clean.columns)}")

print("\nTIMESTAMP")
print(f"Data type: {df_clean['event_time'].dtype}")
print(f"Timezone: {df_clean['event_time'].dt.tz}")

print("\nEVENT TYPES")
print(df_clean["event_type"].value_counts())

print("\nMISSING JOURNEY IDENTIFIERS")
print(
    df_clean[
        ["user_id", "user_session", "product_id"]
    ].isna().sum()
)

print("\n========== VALIDATION FINISHED ==========")


========== FUNNEL INPUT VALIDATION ==========

REQUIRED COLUMNS
user_id: True
user_session: True
product_id: True
event_type: True
event_time: True

DATASET STRUCTURE
Rows: 849,220
Columns: 9

TIMESTAMP
Data type: datetime64[us, UTC]
Timezone: UTC

EVENT TYPES
event_type
view        814847
cart         18938
purchase     15435
Name: count, dtype: int64

MISSING JOURNEY IDENTIFIERS
user_id         0
user_session    0
product_id      0
dtype: int64

========== VALIDATION FINISHED ==========


In [5]:
# Mission 04 — Step 2B
# Reconstruct the exact Phase 1 journey-level funnel

session_events = df_clean[
    [
        "user_id",
        "user_session",
        "product_id",
        "event_time",
        "event_type"
    ]
].dropna(subset=["user_session"]).copy()

session_events["event_time"] = pd.to_datetime(
    session_events["event_time"],
    utc=True
)

keys = ["user_id", "user_session", "product_id"]

# Earliest view in each journey
session_views = (
    session_events.loc[
        session_events["event_type"] == "view"
    ]
    .groupby(keys)["event_time"]
    .min()
    .rename("first_view")
)

# Cart must occur strictly after the view
session_carts = (
    session_events.loc[
        session_events["event_type"] == "cart",
        keys + ["event_time"]
    ]
    .join(session_views, on=keys)
)

valid_session_carts = session_carts.loc[
    session_carts["event_time"] > session_carts["first_view"]
]

first_session_carts = (
    valid_session_carts
    .groupby(keys)["event_time"]
    .min()
    .rename("first_valid_cart")
)

# Purchase must occur strictly after the qualifying cart
session_purchases = (
    session_events.loc[
        session_events["event_type"] == "purchase",
        keys + ["event_time"]
    ]
    .join(first_session_carts, on=keys)
)

valid_session_purchases = session_purchases.loc[
    session_purchases["event_time"]
    > session_purchases["first_valid_cart"]
]

# Recalculate the three journey-level counts
clean_funnel = {
    "view_journeys": len(session_views),
    "view_to_cart_journeys": len(first_session_carts),
    "completed_journeys": (
        valid_session_purchases[keys]
        .drop_duplicates()
        .shape[0]
    )
}

print("========== CLEANED JOURNEY-LEVEL FUNNEL ==========")

for metric, value in clean_funnel.items():
    print(f"{metric}: {value:,}")

========== CLEANED JOURNEY-LEVEL FUNNEL ==========
view_journeys: 552,900
view_to_cart_journeys: 12,725
completed_journeys: 6,300


In [6]:
# Mission 04 — Step 3
# Compare Phase 1 baseline with cleaned funnel

comparison = pd.DataFrame({
    "Phase_1": phase1_baseline,
    "Cleaned": clean_funnel
})

comparison["Difference"] = (
    comparison["Cleaned"] - comparison["Phase_1"]
)

comparison["Match"] = comparison["Difference"] == 0

print("========== FUNNEL RECONCILIATION ==========")
display(comparison)

assert comparison["Match"].all(), (
    "Funnel reconciliation failed!"
)

print("\nPASS: All funnel metrics match Phase 1.")

========== FUNNEL RECONCILIATION ==========


,Phase_1,Cleaned,Difference,Match
view_journeys,552900,552900,0,True
view_to_cart_journeys,12725,12725,0,True
completed_journeys,6300,6300,0,True



PASS: All funnel metrics match Phase 1.
